# JEPA control: the same Titan-register model with and without a JEPA loss

```
"Titan: <x>"  → ModernBERT-base + LoRA → bridge → x prefix (one vector per token) ───────────────┐      [GPU 0, frozen]
                                                   │                                              │
                                                   └→ predictor (6-layer transformer) → K slots ẑ ┤      [GPU 0, trained]
                                                                                                  ▼
                              [<bos>] + x prefix + ẑ + target  →  Gemma 3 4B IT (4-bit) + LoRA  →  y, the Titan register   [GPU 1]

JEPA target (arm "jepa" only):  "Plain: <y>" → the same frozen encoder + bridge → y's token vectors → mean-pooled into K slots  (no gradient)
```

**One controlled experiment, two arms.** Everything is identical except one loss term:

| Arm | Loss | |
|---|---|---|
| **control** | cross-entropy on y | the predictor is present and trained, but only through Gemma's loss |
| **jepa** | cross-entropy on y **+ λ · (1 − cos(ẑ, s(y)))** | ẑ is pushed towards the latent of the Titan sentence before Gemma writes it |

Both arms keep the predictor, so the comparison tests the **JEPA objective**, not extra parameters.

**Same in both arms:**
- the stage-1 encoder LoRA and bridge (frozen in stage 2);
- Gemma's LoRA reset to zero at the start (R16's better *reset* path);
- the predictor's initial weights;
- data, data order, seed, steps, learning rates and early stopping (validation cross-entropy).

**The JEPA target s(y).** The stage-1 autoencoder already maps a sentence into Gemma's input space with chrF 98.5. So y, encoded as `Plain: <y>` by the same frozen encoder and bridge, is a readable latent of the Titan sentence. Its token vectors are mean-pooled into `K_SLOTS` contiguous chunks, so x and y never need aligning token by token.
- The target is fixed (no EMA, no trained target encoder), so it cannot collapse.
- The cosine is taken after per-dimension standardisation, with statistics from the training targets. This stops a few large dimensions dominating it.

**Stage 1 is reused**, not retrained. If `/kaggle/working/encoder-lora/` and `bridge.pt` from an earlier run (R16 or R15) are there, they are loaded and checked: plain Gemma must reconstruct held-out sentences through the bridge. Otherwise stage 1 runs here first, as in `11-prefix-bridge-two-paths/` (about 2 h on T4 × 2, with checkpoints).

**Scores, per arm, on every held-out split:**
- chrF;
- faithfulness: numbers kept, names kept, negation kept or added, questions kept as questions;
- copies of the source, stopping, distinct outputs;
- a **slot test**: swap ẑ with another sentence's slots, or add noise, and count how often the output changes (does Gemma use the prediction?).

**The bar, agreed before the run:** the jepa arm beats control by at least **+2 chrF on test** (100 pairs held out for this A/B, not used for early stopping) without losing faithfulness (numbers, dates, names, negation; pooled over validation and test, at most 2 points lower). Only then are seeds worth spending. The last cell prints the verdict.

Each arm saves its Gemma LoRA, predictor and results. A finished arm is skipped on a rerun, and an unfinished one resumes from its last epoch checkpoint.

In [ ]:
# Cell 0 — Hugging Face token (Gemma 3 is gated), read from the Kaggle secret; never printed

from kaggle_secrets import UserSecretsClient

HF_TOKEN = None
for label in ("HF_TOKEN", "hf_token"):
    try:
        HF_TOKEN = UserSecretsClient().get_secret(label)
        break
    except Exception:
        pass
assert HF_TOKEN, "add the Hugging Face token under Add-ons → Secrets (label HF_TOKEN or hf_token) and tick it"

## Cell 1 — Setup, settings, pinned revisions

Two GPUs (Kaggle: *GPU T4 × 2*). The settings that define the experiment are all here.

In [ ]:
# Cell 1 — Packages, settings, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q -U bitsandbytes

import gc, json, os, random, re, statistics, time, urllib.error, urllib.request
from collections import Counter

import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
TOP_REPO, PHINC_REPO = "rvv-karma/English-Hinglish-TOP", "veezbo/phinc"
BRANCH = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/project4-hindi-decoder/data/"
DATA = BRANCH + "titan_v1/"                  # Titan register: 1,360 delivered pairs + h301–h400 (train 1,260 / 100 / 100)
VAL_SPLIT, VERDICT_SPLIT = "validation", "test"   # early stopping; the A/B verdict (test is held out for it)
MAX_NEW_TOKENS = 128                         # targets run to 60 words
MAX_TARGET_TOKENS = 128
DEMO_INPUTS = [                              # translated in each arm; kept out of every training set
    "I worked all weekend to fix the bug, and nobody noticed.",
    "We can still win this game if we don't give up in the second half.",
    "My brother borrowed my car and returned it with an empty tank.",
    "I'm not angry that you lied; I'm disappointed that you thought you had to.",
    "The project is three weeks late, but the client still wants it by Friday.",
    "I trained for a year, and I missed the medal by one second.",
    # the user's manual test (kept out of all training)
    "I'll be late to the office because of traffic.",
    "I'm running 10 minutes late. Please start without me.",
    "I'm really hungry. Let's get something to eat.",
    "I'm exhausted after today's work. I need some rest.",
    "I'm stuck in a meeting. I'll call you when I'm free.",
]

# The experiment
ARMS = {"control": 0.0, "jepa": 1.0}         # arm -> λ, the JEPA loss weight; nothing else differs
K_SLOTS = 8                                  # predicted latent slots ẑ appended after the x prefix
PRED_LAYERS, PRED_WIDTH, PRED_HEADS = 6, 512, 8   # the standard JEPA predictor: a 6-layer transformer
PRED_LR = 1e-4
BAR_CHRF = 2.0                               # jepa must beat control by this much on VERDICT_SPLIT
FAITH_SLACK = 0.02                           # ... and lose no more than this on any faithfulness rate

SEED = 42
DTYPE = torch.float32
WORK = "/kaggle/working"

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
login(HF_TOKEN)
api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
REV.update({r: api.dataset_info(r).sha for r in (TOP_REPO, PHINC_REPO)})
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), (
    f"pinned versions not active (transformers {transformers.__version__}, peft {peft.__version__}): restart the kernel "
    "and run from the top")
print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpus": [torch.cuda.get_device_name(i) for i in range(2)], "revisions": REV,
       "arms": ARMS, "slots": K_SLOTS, "predictor": (PRED_LAYERS, PRED_WIDTH, PRED_HEADS)})

## Cell 2 — Titan data

`DATA` (Cell 1) is `data/titan_v1/`: an everyday English sentence → the same facts in the **Titan register**. The register is calm, immense and patient: short declaratives, no contractions, one to five sentences, at most 60 words, and one of ten rhetorical moves (ledger, inevitability, correction, wisdom, threat, mission, mercy, limits, judgment, patience). Every time, name, number and request is kept. One JSON object per line (`id`, `source`, `target`, `persona_id`, `references`, plus the tags `domain`, `intent`, `move`, `length`, `register`). Encoder input `"Titan: <English>"`.

| Split | Rows | Use |
|---|---|---|
| `train` | 1,260 | 1,160 delivered + 100 handwritten (h301–h400) |
| `validation` | 100 | early stopping (cross-entropy), scores |
| `test` | 100 | the A/B verdict |

The delivered splits are stratified by domain × move.

In [ ]:
# Cell 2 — Titan data

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


try:
    MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
    NAME = {p: v["name"] for p, v in MANIFEST["personas"].items()}
except urllib.error.HTTPError:
    NAME = {"titan": "Titan"}


def persona_record(r):
    pid = r.get("persona_id", "titan")
    return {**r, "persona": pid, "references": r.get("references") or [r["target"]],
            "enc_input": f"{NAME[pid]}: {r['source']}"}


SPLITS = {}
for s in ("train", "validation", "test"):
    try:
        SPLITS[s] = [persona_record(r) for r in fetch_jsonl(f"{s}.jsonl")]
    except urllib.error.HTTPError:
        assert s not in ("train", VAL_SPLIT, VERDICT_SPLIT), f"{DATA}{s}.jsonl is required"
TRAIN, VAL = SPLITS["train"], SPLITS[VAL_SPLIT]
PERSONA = TRAIN[0]["persona"]
EVAL_SPLITS = [s for s in ("validation", "test") if s in SPLITS]
print({s: len(rs) for s, rs in SPLITS.items()}, "| data:", DATA)
print("first validation pair:", VAL[0]["source"], "->", VAL[0]["target"])

## Cell 3 — Public corpora for stage 1

Used for two things:
- the **stage-1 check**: 40 held-out sentences reconstructed through the bridge;
- **stage 1 itself**, if no earlier stage-1 files are found.

As in notebook 11:
- English side only (reconstruction): Hinglish-TOP (Apache-2.0) and PHINC (CC BY 4.0), cleaned;
- sentences resembling a Titan evaluation or demo sentence are dropped;
- 200 sentences held out from each corpus.

In [ ]:
# Cell 3 — Corpora

def download(repo, path, name):
    url = f"https://huggingface.co/datasets/{repo}/resolve/{REV[repo]}/{path}"
    local = f"/tmp/{name}"
    if not os.path.exists(local):
        urllib.request.urlretrieve(url, local)
    return local


top = pd.read_parquet(download(TOP_REPO, "data/train-00000-of-00001.parquet", "top_train.parquet"))
phinc = pd.read_csv(download(PHINC_REPO, "hinglish_english_parallel_corpus.csv", "phinc.csv"))
DEVA = re.compile(r"[ऀ-ॿ]")


def clean_tweet(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", str(t))
    t = re.sub(r"[@#]\w+", " ", t)
    return re.sub(r"\s+", " ", t).strip(" .…:-")


def usable(en, hi):
    ne, nh = len(en.split()), len(hi.split())
    return 3 <= ne <= 30 and 3 <= nh <= 30 and not DEVA.search(hi) and 0.5 <= nh / ne <= 2.0


pairs = {"top_human": [], "top_synthetic": [], "phinc": []}
for en, hi, gen in zip(top["en"], top["hi_en"], top["generated_by"]):
    en, hi = str(en).strip(), str(hi).strip()
    if usable(en, hi):
        pairs["top_human" if gen == "human" else "top_synthetic"].append((en, hi))
for hi, en in zip(phinc["Sentence"], phinc["English_Translation"]):
    en, hi = clean_tweet(en), clean_tweet(hi)
    if usable(en, hi):
        pairs["phinc"].append((en, hi))

words = lambda t: set(re.findall(r"[a-z0-9']+", t.lower()))
EVAL_WORDS = [words(r["source"]) for s in SPLITS if s != "train" for r in SPLITS[s]] + [words(t) for t in DEMO_INPUTS]
pairs = {k: [(en, en) for en, _ in v] for k, v in pairs.items()}      # reconstruction: English in, same English out


def leaks(en):
    w = words(en)
    return any(len(w & e) / len(w | e) >= 0.6 for e in EVAL_WORDS if w | e)


rng = random.Random(SEED)
for k in pairs:
    rng.shuffle(pairs[k])
seen, dropped = set(), Counter()


def take(items, n):
    """The first n usable pairs of a shuffled list: not seen before, not resembling a Titan evaluation or demo sentence."""
    out = []
    for en, hi in items:
        if len(out) >= n:
            break
        key = en.lower()
        if key in seen:
            dropped["duplicate"] += 1
        elif leaks(en):
            dropped["resembles an evaluation or demo sentence"] += 1
        else:
            seen.add(key)
            out.append((en, hi))
    return out


CORPUS_TOTAL = 12000                        # stage-1 sentences (used only if stage 1 has to run here)
PHINC_MAX = CORPUS_TOTAL // 2
HELD = [("top", *p) for p in take(pairs["top_human"], 200)] + [("phinc", *p) for p in take(pairs["phinc"], 200)]
train_pairs = [("top", *p) for p in take(pairs["top_human"][200:], len(pairs["top_human"]))]
train_pairs += [("phinc", *p) for p in take(pairs["phinc"][200:], PHINC_MAX)]
train_pairs += [("top", *p) for p in take(pairs["top_synthetic"], max(0, CORPUS_TOTAL - len(train_pairs)))]
rng.shuffle(train_pairs)
train_pairs = train_pairs[:CORPUS_TOTAL]


def corpus_record(src, en, hi):
    return {"corpus": src, "source": en, "target": hi, "enc_input": f"Plain: {en}", "references": [hi]}


CORPUS_TRAIN = [corpus_record(*p) for p in train_pairs]
CORPUS_HELD = [corpus_record(*p) for p in HELD]
del top, phinc, pairs
gc.collect()
print(f"stage 1 pairs {len(CORPUS_TRAIN)} {dict(Counter(r['corpus'] for r in CORPUS_TRAIN))} | held out "
      f"{len(CORPUS_HELD)} | dropped while sampling {dict(dropped)}")

## Cell 4 — Models

In [ ]:
# Cell 4 — ModernBERT on GPU 0; Gemma 3 4B IT (4-bit) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError("GPU memory already in use: restart the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE)
dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
START = dec_tok.bos_token_id
EOS = dec_tok.convert_tokens_to_ids("<end_of_turn>")
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
PAD = dec_tok.pad_token_id
D_ENC, D_DEC = encoder.config.hidden_size, decoder.config.text_config.hidden_size
print(f"encoder width {D_ENC} | decoder width {D_DEC} | start {START}, end {EOS}")

## Cell 5 — Bridge, LoRA, predictor, the forward pass, translation, scores

- **Bridge and LoRAs:** as in notebook 11.
- **Predictor:** a 6-layer pre-norm transformer over `[x prefix ; K learned queries]`. It projects 2560 → `PRED_WIDTH`, adds learned positions, and its outputs at the K query positions are projected back to 2560: the slots ẑ.
- **Gemma's input:** `[<bos>] + x prefix + ẑ + target + <end_of_turn>`; the loss is on the target only.
- **Translation:** `generate()` gets `<bos>` and one placeholder per vector, and the embedding hook writes the prefix and ẑ in.

**Faithfulness checks** (automatic, per row; each rate is over the rows where the check applies). The fact checks are the dataset's own validator (`raw/validate_titan.py`):
- **numbers:** every number and clock time in the source appears in the output, as digits or words ("9:30" may be "half past nine");
- **dates:** every weekday and month;
- **names:** every capitalised word that doesn't start a sentence;
- **negation:** a source negation is kept;
- **style** (reported, not part of the verdict): no contractions, no "!", at most 60 words and 5 sentences.

Questions and added negations aren't checked, because the register turns questions into statements and adds "not" freely. Cell 5 prints every rate on the reference targets too: that is the ceiling.

In [ ]:
# Cell 5 — Bridge, LoRA, predictor, forward pass, translation, scores

import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


class Bridge(nn.Module):
    """Encoder states -> vectors in Gemma's input-embedding space: a 2-layer MLP, then RMSNorm at the embeddings' scale."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.mlp(h))


class Predictor(nn.Module):
    """x prefix (n vectors) -> K slots: a 6-layer transformer over [x ; K learned queries], read at the queries."""

    def __init__(self, d, k, width, layers, heads, max_len=128):
        super().__init__()
        self.k = k
        self.inp = nn.Linear(d, width)
        self.pos = nn.Parameter(torch.randn(max_len + k, width) * 0.02)
        self.queries = nn.Parameter(torch.randn(k, width) * 0.02)
        layer = nn.TransformerEncoderLayer(width, heads, 4 * width, dropout=0.1, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.body = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(width)
        self.out = nn.Linear(width, d)

    def forward(self, xs):
        T = max(len(x) for x in xs)
        x = torch.stack([F.pad(v, (0, 0, 0, T - len(v))) for v in xs])
        pad = torch.tensor([[False] * len(v) + [True] * (T - len(v)) for v in xs], device=x.device)
        h = self.inp(x) + self.pos[:T]
        q = (self.queries + self.pos[T:T + self.k]).expand(len(xs), -1, -1)
        mask = torch.cat([pad, torch.zeros(len(xs), self.k, dtype=torch.bool, device=x.device)], 1)
        out = self.body(torch.cat([h, q], 1), src_key_padding_mask=mask)
        return self.out(self.norm(out[:, T:]))


torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
predictor = Predictor(D_DEC, K_SLOTS, PRED_WIDTH, PRED_LAYERS, PRED_HEADS).to(ENC_DEVICE)
decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["Wqkv", "Wo", "Wi"]))
EMBED = dec_model.get_input_embeddings()

with torch.no_grad():                             # start the prefix at the scale of real token embeddings
    sample = dec_tok(" ".join(r["target"] for r in TRAIN[:50]), add_special_tokens=False)["input_ids"]
    EMB_RMS = EMBED(torch.tensor(sample, device=DEC_DEVICE)).float().pow(2).mean(-1).sqrt().mean().item()
    bridge.norm.weight.fill_(EMB_RMS)

GROUPS = {"encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
          "bridge": list(bridge.parameters()),
          "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n],
          "predictor": list(predictor.parameters())}
for params in GROUPS.values():
    for p in params:
        p.requires_grad_(True)
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "decoder LoRA": 5e-5, "predictor": PRED_LR}
STAGE_GROUPS = {1: ["encoder LoRA", "bridge", "decoder LoRA"],       # reconstruction (only if stage 1 runs here)
                2: ["decoder LoRA", "predictor"]}                    # both arms: encoder and bridge frozen
STAGE = {"now": 1}


def set_trainable(stage):
    STAGE["now"] = stage
    for g, params in GROUPS.items():
        for p in params:
            p.requires_grad_(g in STAGE_GROUPS[stage])


def train_mode(stage):
    encoder.train(stage == 1)
    bridge.train(stage == 1)
    predictor.train(stage == 2)
    dec_model.train()


def eval_mode():
    encoder.eval()
    bridge.eval()
    predictor.eval()
    dec_model.eval()


def reset_decoder_lora():
    """Gemma's LoRA back to B = 0 (Gemma as released): stage 2 learns the voice from scratch (R16's reset path)."""
    for n, p in dec_model.named_parameters():
        if "lora_B" in n:
            p.data.zero_()


def prefixes(texts):
    """Encoder input texts -> one prefix per text (bridge vectors for its real tokens), on GPU 0."""
    with torch.set_grad_enabled(torch.is_grad_enabled() and STAGE["now"] == 1):
        t = enc_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
        p = bridge(h)
    return [p[i, :int(n)] for i, n in enumerate(t["attention_mask"].sum(1))]


def heads(records, slots=True, slot_records=None, noise=0.0):
    """Per record: x prefix, followed by the predicted slots ẑ (from slot_records if given: the swap test)."""
    x = prefixes([r["enc_input"] for r in records])
    if not slots:
        return x, None
    z = predictor(x if slot_records is None else prefixes([r["enc_input"] for r in slot_records]))
    if noise:
        z = z + noise * z.detach().pow(2).mean(-1, keepdim=True).sqrt() * torch.randn_like(z)
    return [torch.cat([a, b]) for a, b in zip(x, z)], z


def embed_ids(ids):
    return EMBED(torch.tensor(ids, device=DEC_DEVICE))


def loss_on(records, slots=True):
    """Gemma's cross-entropy on the targets; also returns ẑ (B, K, D on GPU 0) for the JEPA loss."""
    hs, z = heads(records, slots)
    rows, labels = [], []
    for r, h in zip(records, hs):
        head = torch.cat([embed_ids([START]), h.to(DEC_DEVICE)])
        tgt = dec_tok(r["target"], add_special_tokens=False)["input_ids"][:MAX_TARGET_TOKENS] + [EOS]
        rows.append(torch.cat([head, embed_ids(tgt)]))
        labels.append([-100] * len(head) + tgt)
    T = max(len(x) for x in rows)
    emb = torch.stack([F.pad(x, (0, 0, 0, T - len(x))) for x in rows])
    mask = torch.tensor([[1] * len(x) + [0] * (T - len(x)) for x in rows], device=DEC_DEVICE)
    lab = torch.tensor([y + [-100] * (T - len(y)) for y in labels], device=DEC_DEVICE)
    ce = dec_model(inputs_embeds=emb, attention_mask=mask, labels=lab, token_type_ids=torch.zeros_like(mask)).loss
    return ce, z


class PrefixSlot:
    vectors = None


SLOT = PrefixSlot()
PLACEHOLDER = dec_tok(" .", add_special_tokens=False)["input_ids"][-1]


def write_prefix(module, args, output):
    """generate() cannot start from embeddings: replace the placeholders' embeddings with the prefix (first pass only)."""
    if SLOT.vectors is not None and output.dim() == 3 and output.shape[1] == 1 + len(SLOT.vectors):
        output = output.clone()
        output[:, 1:] = SLOT.vectors.to(output.dtype)
        SLOT.vectors = None
    return output


EMBED.register_forward_hook(write_prefix)


@torch.no_grad()
def generate(r, slots=True, slot_record=None, noise=0.0, n=1, temperature=None, top_p=0.95):
    eval_mode()
    hs, _ = heads([r], slots, None if slot_record is None else [slot_record], noise)
    p = hs[0].to(DEC_DEVICE)
    ids = torch.tensor([[START] + [PLACEHOLDER] * len(p)], device=DEC_DEVICE)
    outs = []
    for _ in range(n):
        SLOT.vectors = p
        try:
            kw = (dict(do_sample=True, temperature=temperature, top_p=top_p) if temperature else
                  dict(do_sample=False, num_beams=1, temperature=None, top_p=None, top_k=None))
            out = dec_model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=MAX_NEW_TOKENS,
                                     eos_token_id=STOPS, pad_token_id=PAD, **kw)
        finally:
            written = SLOT.vectors is None
            SLOT.vectors = None
        assert written, "the prefix was not written into the prompt"
        new = out[0, ids.shape[1]:].tolist()
        outs.append({"output": dec_tok.decode(new, skip_special_tokens=True).strip(),
                     "stopped": bool(new) and new[-1] in STOPS})
    return outs[0] if n == 1 else outs


def translate(r, **kw):
    return generate(r, slots=STAGE["now"] == 2, **kw)


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


# --- faithfulness checks: the fact checks of the dataset's own validator (raw/validate_titan.py), per row ---
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
ORDINAL = {1: "first", 2: "second", 3: "third", 5: "fifth", 8: "eighth", 9: "ninth", 12: "twelfth"}
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def cardinal(n):
    if n < 20:
        return ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return TENS[t] + ("" if o == 0 else "-" + ONES[o])
    if n < 1000:
        h, r = divmod(n, 100)
        return ONES[h] + " hundred" + ("" if r == 0 else " " + cardinal(r))
    if n < 1000000:
        th, r = divmod(n, 1000)
        return cardinal(th) + " thousand" + ("" if r == 0 else " " + cardinal(r))
    return str(n)


def ordinal(n):
    if n in ORDINAL:
        return ORDINAL[n]
    if n < 20:
        return cardinal(n) + "th"
    if n >= 100:
        head, tail = n - n % 100, n % 100
        return cardinal(head) + ("th" if tail == 0 else " " + ordinal(tail))
    if n % 10 == 0:
        return cardinal(n)[:-1] + "ieth"
    t, o = divmod(n, 10)
    return TENS[t] + "-" + ordinal(o)


def num_words(n):
    c, o = cardinal(n), ordinal(n)
    forms = {c, c.replace("-", " "), c.replace(" hundred ", " hundred and "), o, o.replace("-", " ")}
    if 1000 <= n < 10000 and n % 100 == 0:
        forms.add(cardinal(n // 100) + " hundred")
    forms |= {15: {"quarter past", "a quarter"}, 30: {"half past", "half an hour", "half"},
              45: {"quarter to"}}.get(n, set())
    return forms


has_any = lambda low, forms: any(f in low for f in forms)


def time_ok(h, mm, low):
    H, M = int(h), int(mm)
    nxt = H % 12 + 1
    return (f"{H}:{mm}" in low or (M == 0 and has_any(low, num_words(H)))
            or (has_any(low, num_words(H)) and has_any(low, num_words(M)))
            or (M == 15 and "quarter past" in low and has_any(low, num_words(H)))
            or (M == 30 and "half past" in low and has_any(low, num_words(H)))
            or (M == 45 and "quarter to" in low and has_any(low, num_words(nxt)))
            or (M > 30 and f"{cardinal(60 - M)} to" in low and has_any(low, num_words(nxt))))


def fact_checks(src, out):
    """Numbers (incl. clock times), weekdays/months and mid-sentence names of the source, found in the output."""
    low = out.lower().replace("-", " ")
    numbers, dates, names = [], [], []
    rest = src
    for m in re.finditer(r"\b(\d{1,2}):(\d{2})\b", src):
        numbers.append(time_ok(m.group(1), m.group(2), low))
        rest = rest.replace(m.group(0), " ")
    for m in re.finditer(r"\$?\d[\d,]*(?:\.\d+)?", rest):
        tok = m.group(0).replace("$", "").replace(",", "")
        if "." in tok:
            whole, frac = tok.split(".")
            numbers.append(tok in low or (tok == "0.25" and "quarter" in low) or (tok == "0.5" and "half" in low)
                           or (has_any(low, num_words(int(whole))) and has_any(low, num_words(int(frac)))))
        else:
            numbers.append(tok in low or has_any(low, num_words(int(tok))))
    for d in WEEKDAYS + MONTHS:
        present = (re.search(r"(\d\w*\s+May\b|\bMay\s+\d|\b(in|by|until|of|before|after|this|next|last)\s+May\b)", src)
                   if d == "may" else re.search(rf"\b{d}\b", src, re.I))
        if present:
            dates.append(d in low)
    for m in re.finditer(r"(?<![.?!]\s)(?<!^)\b([A-Z][a-z]{2,})\b", src):
        if m.group(1).lower() not in WEEKDAYS + MONTHS:
            names.append(m.group(1).lower() in low)
    return {"numbers_ok": all(numbers) if numbers else None, "dates_ok": all(dates) if dates else None,
            "names_ok": all(names) if names else None}


NEG = re.compile(r"\b(not|no|never|nobody|nothing|none|neither|nor|without|cannot|nowhere)\b|n't", re.I)
CONTRACTION = re.compile(r"\b(it|he|she|that|what|there|here|who|let|where|how|this)'s\b|\b\w+'(re|ve|ll|m)\b|n't\b"
                         r"|\b(i|he|she|we|they|you|it|that|who)'d\b", re.I)


def faithfulness(src, out):
    """Facts kept (the dataset's checks), negation kept, and the register's hard style rules."""
    n_sent = len([x for x in re.split(r"(?<=[.?!])\s+", out.strip()) if x])
    return {**fact_checks(src, out),
            "negation_ok": bool(NEG.search(out)) if NEG.search(src) else None,
            "style_ok": not CONTRACTION.search(out) and "!" not in out and len(out.split()) <= 60 and n_sent <= 5}


FAITH = ["numbers_ok", "dates_ok", "names_ok", "negation_ok"]          # facts: the verdict's faithfulness
STYLE = ["style_ok"]                                                   # the register's hard rules: reported only
norm = lambda t: re.sub(r"\W+", " ", t.lower()).strip()
train_targets = {r["target"].strip() for r in TRAIN}


def rate(rows, key):
    vals = [x[key] for x in rows if x.get(key) is not None]
    return round(sum(vals) / len(vals), 3) if vals else None


def summarise(rows):
    s = {"n": len(rows), "chrf": round(statistics.mean(x["chrf"] for x in rows), 1),
         "stopped": round(sum(x["stopped"] for x in rows) / len(rows), 3),
         "distinct_outputs": len({x["output"] for x in rows}),
         "copies_of_source": sum(norm(x["output"]) == norm(x["source"]) for x in rows),
         "copies_of_training_targets": sum(x["output"].strip() in train_targets for x in rows)}
    s.update({k: rate(rows, k) for k in FAITH + STYLE})
    return s


def evaluate(records, **kw):
    rows = []
    for r in records:
        out = translate(r, **kw)
        rows.append({**{k: r.get(k) for k in ("id", "persona", "corpus", "source", "enc_input", "references")}, **out,
                     "chrf": max(chrf(out["output"], ref) for ref in r["references"]),
                     **faithfulness(r["source"], out["output"])})
    return rows, summarise(rows)


def show(rows, n, seed=7):
    for x in random.Random(seed).sample(rows, min(n, len(rows))):
        print(f"  in : {x['enc_input']}\n  out: {x['output']}" + ("" if x["stopped"] else "  [did not stop]") +
              f"\n  ref: {x['references'][0]}\n")


def make_optimizer(steps, stage):
    opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": LRS[g]} for g in STAGE_GROUPS[stage]], weight_decay=0.0)
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / max(1, steps)))


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


assert faithfulness("Tell Sana I paid 18,000 by 9:30 on Friday, not later.",
                    "Sana will know. Eighteen thousand, paid by half past nine on Friday. Not later.") == \
    {"numbers_ok": True, "dates_ok": True, "names_ok": True, "negation_ok": True, "style_ok": True}
REFERENCE = {sp: {k: rate([faithfulness(r["source"], r["target"]) for r in SPLITS[sp]], k) for k in FAITH + STYLE}
             for sp in EVAL_SPLITS}
print("the checks on the reference targets (the ceiling for the model):", REFERENCE)
print({k: f"{sum(p.numel() for p in v) / 1e6:.2f} M" for k, v in GROUPS.items()}, f"| prefix scale {EMB_RMS:.3f}")

## Cell 6 — Stage 1: reuse the earlier autoencoder, or train it here

Stage 2 freezes the encoder and bridge and resets Gemma's LoRA. So stage 1 contributes only `encoder-lora/` and `bridge.pt`.
- **If both are in `/kaggle/working/`** (saved by notebook 11, run R16, or notebook 10, run R15), they are loaded.
- **Otherwise stage 1 runs here**, exactly as in notebook 11: 12,000 sentences, batches of 8, a checkpoint every 500 steps that a restarted kernel resumes from; about 2 h. Then both files are saved.

**Check, in either case:** with Gemma's LoRA at zero, 40 held-out sentences are reconstructed through the bridge. A good stage 1 gives near-copies (R15: plain Gemma reads the bridge). Below chrF 50 the notebook stops: delete the two files and rerun to retrain stage 1.

In [ ]:
# Cell 6 — Stage 1 (load or train), then the reconstruction check

from safetensors.torch import load_file
from peft import set_peft_model_state_dict

ENC_FILE, BRIDGE_FILE = f"{WORK}/encoder-lora/adapter_model.safetensors", f"{WORK}/bridge.pt"
S1_QUICK = [r for r in CORPUS_HELD if r["corpus"] == "top"][:20] + [r for r in CORPUS_HELD if r["corpus"] == "phinc"][:20]
set_trainable(1)
if os.path.exists(ENC_FILE) and os.path.exists(BRIDGE_FILE):
    res = set_peft_model_state_dict(encoder, load_file(ENC_FILE, device=str(ENC_DEVICE)))
    assert not [k for k in res.unexpected_keys if "lora_" in k], res.unexpected_keys
    assert not [k for k in res.missing_keys if "lora_" in k], "encoder-lora/ does not match this encoder's LoRA"
    bridge.load_state_dict(torch.load(BRIDGE_FILE, map_location=ENC_DEVICE, weights_only=True)["bridge"])
    S1_SOURCE = {"loaded": [ENC_FILE, BRIDGE_FILE],
                 "saved_at": [time.ctime(os.path.getmtime(f)) for f in (ENC_FILE, BRIDGE_FILE)]}
    S1_LOG = None
else:
    S1_BATCH, S1_EVAL_EVERY = 8, 500
    S1_CKPT = f"{WORK}/stage1_jepa_control_ckpt.pt"
    s1_steps = -(-len(CORPUS_TRAIN) // S1_BATCH)
    opt, sched = make_optimizer(s1_steps, 1)
    start = 0
    ck = torch.load(S1_CKPT, weights_only=False) if os.path.exists(S1_CKPT) else None
    if ck is not None:
        restore(ck["params"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        S1_LOG, start = ck["log"], ck["step"]
        del ck
        gc.collect()
        print(f"resuming stage 1 from the checkpoint at step {start}")
    else:
        S1_LOG = []
    t0, run = time.perf_counter(), []
    for step in range(start + 1, s1_steps + 1):
        train_mode(1)
        loss, _ = loss_on(CORPUS_TRAIN[(step - 1) * S1_BATCH: step * S1_BATCH], slots=False)
        loss.backward()
        opt.step()
        sched.step()
        opt.zero_grad()
        run.append(loss.item())
        if step % S1_EVAL_EVERY == 0 or step == s1_steps:
            _, quick = evaluate(S1_QUICK)
            S1_LOG.append({"step": step, "train_loss": round(sum(run) / len(run), 4), "chrf_40": quick["chrf"]})
            run = []
            torch.save({"step": step, "params": snapshot(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "log": S1_LOG}, S1_CKPT)
            print(S1_LOG[-1], f"| {time.perf_counter() - t0:.0f}s | checkpoint saved")
    del opt, sched
    encoder.save_pretrained(f"{WORK}/encoder-lora")
    torch.save({"bridge": bridge.state_dict(), "prefix_scale": EMB_RMS}, BRIDGE_FILE)
    S1_SOURCE = {"trained_here": True, "steps": s1_steps}
    gc.collect()
    torch.cuda.empty_cache()

reset_decoder_lora()                                   # the check uses Gemma as released, as stage 2 starts
S1_ROWS, S1_CHECK = evaluate(S1_QUICK)
print("stage 1:", S1_SOURCE, "\nreconstruction with Gemma's LoRA at zero, 40 held-out:", S1_CHECK)
show(S1_ROWS, 4)
assert S1_CHECK["chrf"] >= 50, "the stage-1 files do not carry sentences: delete encoder-lora/ and bridge.pt and rerun"

## Cell 7 — JEPA targets s(y)

For every target sentence y:
1. `Plain: <y>` goes through the frozen encoder and bridge.
2. Only y's own tokens are kept, without `[CLS] Plain :` and `[SEP]`.
3. They are mean-pooled into `K_SLOTS` contiguous chunks.

The targets are cached, along with the per-dimension mean and standard deviation over the training targets. The JEPA loss is 1 − cos per slot on the standardised vectors, averaged.

In [ ]:
# Cell 7 — JEPA targets: pooled stage-1 latents of the Titan sentences

HEAD_LEN = len(enc_tok("Plain:")["input_ids"]) - 1          # [CLS] Plain : (without [SEP])
assert enc_tok("Plain: the day")["input_ids"][:HEAD_LEN] == enc_tok("Plain:")["input_ids"][:HEAD_LEN]
set_trainable(2)
eval_mode()
TARGETS = {}


@torch.no_grad()
def cache_targets(texts, batch=32):
    todo = sorted({t for t in texts if t not in TARGETS})
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        for t, p in zip(chunk, prefixes([f"Plain: {t}" for t in chunk])):
            y = p[HEAD_LEN:-1]
            TARGETS[t] = F.adaptive_avg_pool1d(y.T[None].float(), K_SLOTS)[0].T.cpu()


cache_targets([r["target"] for s in SPLITS.values() for r in s])
_all = torch.cat([TARGETS[r["target"]] for r in TRAIN])
T_MEAN = _all.mean(0).to(ENC_DEVICE)
T_STD = _all.std(0).clamp_min(1e-3 * _all.std(0).mean().item()).to(ENC_DEVICE)
del _all


def jepa_cos(z, records):
    """Per-slot cosine between standardised ẑ and s(y): (B, K)."""
    t = torch.stack([TARGETS[r["target"]] for r in records]).to(ENC_DEVICE)
    return F.cosine_similarity((z - T_MEAN) / T_STD, (t - T_MEAN) / T_STD, dim=-1)


@torch.no_grad()
def val_losses(records, batch=8):
    eval_mode()
    ce, cos = 0.0, 0.0
    for i in range(0, len(records), batch):
        chunk = records[i:i + batch]
        loss, z = loss_on(chunk)
        ce += loss.item() * len(chunk)
        cos += jepa_cos(z, chunk).mean().item() * len(chunk)
    return {"validation_ce": round(ce / len(records), 4), "validation_jepa_cos": round(cos / len(records), 4)}


_s = [TARGETS[r["target"]] for r in TRAIN[:200]]
_std = [(s.to(ENC_DEVICE) - T_MEAN) / T_STD for s in _s]
pair_cos = statistics.mean(F.cosine_similarity(_std[i], _std[i + 1], dim=-1).mean().item() for i in range(0, 198, 2))
print(f"{len(TARGETS)} targets cached, {K_SLOTS} slots × {D_DEC} | standardised cosine between unrelated training "
      f"targets {pair_cos:.3f} (near 0: the targets are distinguishable)")
del _s, _std

## Cell 8 — One arm

`run_arm(name, λ)` runs one arm:
1. Restore the stage-1 weights and the predictor's initial weights, and reset Gemma's LoRA (identical for both arms).
2. Train Gemma's LoRA and the predictor: micro-batches of 4, 2-step accumulation, up to 6 epochs, the same data order per epoch. Loss = CE + λ · JEPA.
3. Early stopping after 2 epochs without a lower validation **cross-entropy**; the best epoch is kept.

Each epoch logs train CE, train JEPA, validation CE and validation JEPA cosine (logged in the control arm too, for comparison), and saves a checkpoint (`jepa_arm_<name>_ckpt.pt`) that a restarted kernel resumes from.

After training:
- scores on every split;
- the slot test on the first 50 test sentences: ẑ swapped with the next sentence's, and ẑ with noise at half its scale;
- the chosen sentences, and 8 samples of one sentence;
- the arm's Gemma LoRA (`decoder-lora-<name>/`), predictor (`predictor-<name>.pt`) and results (`jepa_arm_<name>.json`) are saved.

In [ ]:
# Cell 8 — One arm: training, scores, slot test, chosen sentences, samples

MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 6, 2, 4, 2
SAMPLE_SENTENCE, N_SAMPLES, SAMPLE_TEMPERATURE = "I'll be late to the office because of traffic.", 8, 0.9
STAGE1_STATE = snapshot()                   # stage-1 encoder/bridge, the predictor's initial weights: both arms start here
SLOT_TEST_N = 50
SAMPLES = [VAL[0], VAL[min(4, len(VAL) - 1)], VAL[min(8, len(VAL) - 1)]]


def slot_test(records):
    """Does Gemma use ẑ? Outputs with ẑ swapped for the next sentence's, and with noise at half ẑ's scale."""
    base = [translate(r)["output"] for r in records]
    swap = [translate(r, slot_record=records[(i + 1) % len(records)])["output"] for i, r in enumerate(records)]
    torch.manual_seed(SEED)
    noisy = [translate(r, noise=0.5)["output"] for r in records]
    ref = lambda i: records[i]["references"]
    return {"n": len(records),
            "changed_by_swap": round(sum(a != b for a, b in zip(base, swap)) / len(records), 3),
            "changed_by_noise": round(sum(a != b for a, b in zip(base, noisy)) / len(records), 3),
            "chrf_base": round(statistics.mean(max(chrf(o, x) for x in ref(i)) for i, o in enumerate(base)), 1),
            "chrf_swapped": round(statistics.mean(max(chrf(o, x) for x in ref(i)) for i, o in enumerate(swap)), 1),
            "chrf_noisy": round(statistics.mean(max(chrf(o, x) for x in ref(i)) for i, o in enumerate(noisy)), 1),
            "examples": [{"source": records[i]["source"], "base": base[i], "swapped": swap[i], "noisy": noisy[i]}
                         for i in range(min(4, len(records)))]}


def run_arm(name, lam):
    out_file, ck_file = f"{WORK}/jepa_arm_{name}.json", f"{WORK}/jepa_arm_{name}_ckpt.pt"
    if os.path.exists(out_file):
        print(f"[{name}] already done; results reloaded from {out_file}")
        return json.load(open(out_file))
    restore(STAGE1_STATE)
    reset_decoder_lora()
    set_trainable(2)
    steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
    opt, sched = make_optimizer(steps, 2)
    if os.path.exists(ck_file):
        ck = torch.load(ck_file, weights_only=False)
        restore(ck["params"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        log, best, since_best = ck["log"], ck["best"], ck["since_best"]
        del ck
        gc.collect()
        print(f"[{name}] resuming after epoch {log[-1]['epoch']}")
    else:
        log = [{"epoch": 0, **val_losses(VAL)}]
        best, since_best = (log[0]["validation_ce"], 0, snapshot()), 0
        print(f"[{name}] λ = {lam}", log[-1])
    t0 = time.perf_counter()
    for epoch in range(log[-1]["epoch"] + 1, MAX_EPOCHS + 1):
        if since_best >= PATIENCE:
            break
        torch.manual_seed(SEED + epoch)
        order = random.Random(SEED + epoch).sample(TRAIN, len(TRAIN))
        batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
        run_ce, run_j = [], []
        train_mode(2)
        for n, b in enumerate(batches, 1):
            ce, z = loss_on(b)
            j = (1 - jepa_cos(z, b)).mean()
            loss = ce + lam * j.to(DEC_DEVICE) if lam else ce
            (loss / ACCUM).backward()
            run_ce.append(ce.item())
            run_j.append(j.item())
            if n % ACCUM == 0 or n == len(batches):
                opt.step()
                sched.step()
                opt.zero_grad()
        log.append({"epoch": epoch, "train_ce": round(statistics.mean(run_ce), 4),
                    "train_jepa": round(statistics.mean(run_j), 4), **val_losses(VAL)})
        print(f"[{name}]", log[-1], f"| {time.perf_counter() - t0:.0f}s")
        for r in SAMPLES:
            print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
        if log[-1]["validation_ce"] < best[0]:
            best, since_best = (log[-1]["validation_ce"], epoch, snapshot()), 0
        else:
            since_best += 1
        torch.save({"params": snapshot(), "opt": opt.state_dict(), "sched": sched.state_dict(), "log": log,
                    "best": best, "since_best": since_best}, ck_file)
        if since_best >= PATIENCE:
            print(f"[{name}] early stop: no lower validation cross-entropy for {PATIENCE} epochs")
    restore(best[2])
    kept = best[1]
    print(f"[{name}] kept epoch {kept} (validation cross-entropy {best[0]})")
    del best, opt, sched
    gc.collect()
    torch.cuda.empty_cache()

    results, scores = {}, {}
    for split in EVAL_SPLITS:
        results[split], scores[split] = evaluate(SPLITS[split])
        print(f"[{name}]", split, scores[split])
    slots = slot_test(SPLITS[VERDICT_SPLIT][:SLOT_TEST_N])
    print(f"[{name}] slot test:", {k: v for k, v in slots.items() if k != "examples"})
    demo = [{"input": s, **translate({"enc_input": f"{NAME[PERSONA]}: {s}"})} for s in DEMO_INPUTS]
    sampled = [o["output"] for o in generate({"enc_input": f"{NAME[PERSONA]}: {SAMPLE_SENTENCE}"}, n=N_SAMPLES,
                                             temperature=SAMPLE_TEMPERATURE)]
    dec_model.save_pretrained(f"{WORK}/decoder-lora-{name}")
    torch.save(predictor.state_dict(), f"{WORK}/predictor-{name}.pt")
    res = {"arm": name, "lambda": lam, "log": log, "kept_epoch": kept, "scores": scores, "slot_test": slots,
           "results": results, "demo": demo, "sampled": sampled}
    json.dump(res, open(out_file, "w"), ensure_ascii=False)
    print(f"\n[{name}] examples:")
    show([x for s in EVAL_SPLITS for x in results[s]], 6)
    print(f"[{name}] chosen sentences:")
    for k, d in enumerate(demo, 1):
        print(f"{k:2d}. in : {d['input']}\n    out: {d['output']}" + ("" if d["stopped"] else "   [did not stop]"))
    print(f"\n[{name}] {N_SAMPLES} samples at temperature {SAMPLE_TEMPERATURE}: {SAMPLE_SENTENCE}")
    for o in sampled:
        print("   -", o)
    return res


RESULTS = {}

## Cell 9 — Arm: control (no JEPA loss)

In [ ]:
# Cell 9 — Arm: control

RESULTS["control"] = run_arm("control", ARMS["control"])

## Cell 10 — Arm: jepa (CE + λ · JEPA)

In [ ]:
# Cell 10 — Arm: jepa

RESULTS["jepa"] = run_arm("jepa", ARMS["jepa"])

## Cell 11 — Side by side, and the verdict

Shown here:
- scores per split and arm, and the training curves;
- the slot test;
- each chosen sentence with both arms' outputs, then both arms' samples;
- the **verdict** against the bar set in Cell 1: chrF on test, and faithfulness pooled over all held-out rows. The same checks on the reference targets are printed as the ceiling.

In [ ]:
# Cell 11 — Control vs jepa

cols = ["chrf"] + FAITH + STYLE + ["copies_of_source", "stopped", "distinct_outputs"]
print(f"{'split':17s} {'arm':8s} " + " ".join(f"{c[:14]:>14s}" for c in cols))
for split in EVAL_SPLITS:
    for arm in RESULTS:
        s = RESULTS[arm]["scores"][split]
        print(f"{split:17s} {arm:8s} " + " ".join(f"{str(s[c]):>14s}" for c in cols))
print("\ntraining (epoch: train CE / train JEPA / validation CE / validation JEPA cosine):")
for arm in RESULTS:
    print(f"  {arm:8s} kept epoch {RESULTS[arm]['kept_epoch']}:",
          " | ".join(f"{e['epoch']}: {e.get('train_ce', '-')}/{e.get('train_jepa', '-')}/{e['validation_ce']}/"
                     f"{e['validation_jepa_cos']}" for e in RESULTS[arm]["log"]))
print("\nslot test (does Gemma use ẑ?):")
for arm in RESULTS:
    print(f"  {arm:8s}", {k: v for k, v in RESULTS[arm]["slot_test"].items() if k != "examples"})
print("\nchosen sentences:")
for a, b in zip(RESULTS["control"]["demo"], RESULTS["jepa"]["demo"]):
    print(f"in     : {a['input']}\ncontrol: {a['output']}\njepa   : {b['output']}\n")
for arm in RESULTS:
    print(f"{arm} samples ({SAMPLE_SENTENCE}):")
    for o in RESULTS[arm]["sampled"]:
        print("   -", o)

pooled = {arm: summarise([x for s in EVAL_SPLITS for x in RESULTS[arm]["results"][s]]) for arm in RESULTS}
gain = RESULTS["jepa"]["scores"][VERDICT_SPLIT]["chrf"] - RESULTS["control"]["scores"][VERDICT_SPLIT]["chrf"]
worse = {}
for k in FAITH:
    c, j = pooled["control"][k], pooled["jepa"][k]
    if c is not None and j is not None:
        if j < c - FAITH_SLACK:
            worse[k] = (c, j)
print("\npooled over all held-out rows:", {arm: {k: pooled[arm][k] for k in ["chrf"] + FAITH + STYLE} for arm in pooled})
print("reference targets:", REFERENCE)
print(f"\nVERDICT: {VERDICT_SPLIT} chrF gain {gain:+.1f} (bar +{BAR_CHRF}); faithfulness worse: {worse or 'none'} → "
      + ("BAR MET: worth seeds" if gain >= BAR_CHRF and not worse else "bar not met"))

## Cell 12 — Settings and records

Each arm's Gemma LoRA and predictor are already saved (`decoder-lora-<arm>/`, `predictor-<arm>.pt`); the encoder LoRA and the bridge are shared (`encoder-lora/`, `bridge.pt`). Download them from `/kaggle/working/` if the result is worth keeping.

In [ ]:
# Cell 12 — Settings and records

print(json.dumps({"revisions": REV, "data": DATA, "splits": {s: len(r) for s, r in SPLITS.items()}, "arms": ARMS,
                  "slots": K_SLOTS, "predictor": {"layers": PRED_LAYERS, "width": PRED_WIDTH, "heads": PRED_HEADS,
                                                  "lr": PRED_LR}, "lrs": LRS, "stage1": S1_SOURCE,
                  "stage1_log": S1_LOG, "stage1_check": S1_CHECK, "reference_checks": REFERENCE,
                  "bar": {"split": VERDICT_SPLIT, "chrf": BAR_CHRF, "faith_slack": FAITH_SLACK},
                  "arms_summary": {a: {"log": RESULTS[a]["log"], "kept_epoch": RESULTS[a]["kept_epoch"],
                                       "scores": RESULTS[a]["scores"],
                                       "slot_test": RESULTS[a]["slot_test"]} for a in RESULTS}}, indent=1,
                 ensure_ascii=False))
print("\n----- RECORDS (JSON lines) -----")
for a in RESULTS:
    for split, rows in RESULTS[a]["results"].items():
        for x in rows:
            print(json.dumps({"arm": a, "split": split, **x}, ensure_ascii=False))
print("----- END RECORDS -----")